# Phase 4 — Controlled 10-language pilot (Kaggle)

The controlled roster is English (`en`), German (`de`), Spanish (`es`), French (`fr`), Swahili (`sw`), Arabic (`ar`), Urdu (`ur`), Bulgarian (`bg`), Russian (`ru`), and Hindi (`hi`). This notebook runs the selected method(s) for one frozen order/seed pair. The seed is selected from the protocol. Set `METHOD_IDS` and `ORDER_ID` below; repeat for all five orders and B0–B8 plus M3. The complete pilot is 50 runs.

**Do not start training until the matching-roster Phase 3 v2 analysis selects `published_interference`.** This prior includes the pre-specified k=5 URIEL+ imputation for Arabic, which must be reviewed and recorded. Update the protocol gate only after that review. Five paired orders are a descriptive pilot, not a significance-backed headline study.

Before running: enable Kaggle Internet and a GPU, accept the gated model terms for `meta-llama/Llama-3.2-1B`, and add a Kaggle secret named `HF_TOKEN`. Save the notebook output ZIP after each session so completed result records persist.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Mani-S-2709/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
elif (REPO_ROOT / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
if not (REPO_ROOT / "phase4" / "run_phase4.py").is_file():
    raise FileNotFoundError("The cloned repository is missing Phase 4 files. Push the prepared code to Mani-S-2709, then restart this notebook.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "phase2" / "requirements-phase2.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "urielplus==1.3.3"], check=True)
sys.path.insert(0, str(REPO_ROOT))

from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN Kaggle secret is empty.")
os.environ["HF_TOKEN"] = hf_token
del hf_token
print({"repo": str(REPO_ROOT), "origin": REPO_URL})

In [ ]:
import json
from phase4.run_phase4 import run_selected_pair

PROTOCOL_PATH = REPO_ROOT / "phase4" / "configs" / "controlled_10lang_pilot_protocol.json"
ARTIFACT_ROOT = Path("/kaggle/working/phase4_controlled_10lang_pilot")
MATRIX_PATH = REPO_ROOT / "phase4" / "artifacts" / "interference_matrix_10lang.csv"
ORDER_ID = "order-1"
METHOD_IDS = ["B0"]

protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
expected_languages = ["en", "de", "es", "fr", "sw", "ar", "ur", "bg", "ru", "hi"]
if protocol["dataset"]["languages"] != expected_languages:
    raise ValueError("Phase 4 protocol language roster differs from the approved ten-language stream.")
gate = protocol["phase4"]["prior_gate"]
if gate["status"] != "passed" or gate["selected_prior"] != gate["required_prior"]:
    raise RuntimeError("Phase 4 is blocked until Phase 3 confirms published_interference and the protocol gate is updated.")
if not METHOD_IDS or len(set(METHOD_IDS)) != len(METHOD_IDS):
    raise ValueError("Select one or more distinct Phase 4 methods.")

if "M3" in METHOD_IDS:
    subprocess.run([
        sys.executable, "-m", "phase4.prepare_interference_matrix",
        "--output", str(MATRIX_PATH),
    ], cwd=REPO_ROOT, check=True)

records = []
for method_id in METHOD_IDS:
    result = run_selected_pair(
        method_id=method_id,
        order_id=ORDER_ID,
        protocol_path=PROTOCOL_PATH,
        artifact_root=ARTIFACT_ROOT,
        interference_matrix_path=MATRIX_PATH,
    )
    records.extend(result if isinstance(result, list) else [result])
print("Completed run IDs:", [record["run_id"] for record in records])

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from phase2.protocol import build_manifest, load_protocol
from phase2.results import aggregate_results
from phase2.run_baseline import validate_prepared_artifacts

protocol = load_protocol(PROTOCOL_PATH)
prepared = validate_prepared_artifacts(protocol, ARTIFACT_ROOT)
manifest = build_manifest(protocol)
manifest["data_artifacts"] = prepared["data_artifacts"]
manifest["data_manifest_sha256"] = prepared["data_manifest_sha256"]
aggregate = aggregate_results(manifest, ARTIFACT_ROOT / "results")
archive_path = Path("/kaggle/working") / f"phase4_{ORDER_ID}_{'_'.join(METHOD_IDS)}.zip"
with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    archive.write(ARTIFACT_ROOT / "protocol_manifest.json", "protocol_manifest.json")
    for record in records:
        result_path = ARTIFACT_ROOT / "results" / f"{record['run_id']}.json"
        archive.write(result_path, f"results/{result_path.name}")
    archive.writestr("aggregate.json", json.dumps(aggregate, indent=2, sort_keys=True) + "\n")
    if "M3" in METHOD_IDS:
        archive.write(MATRIX_PATH, f"priors/{MATRIX_PATH.name}")
        metadata_path = MATRIX_PATH.with_name(f"{MATRIX_PATH.stem}_metadata.json")
        archive.write(metadata_path, f"priors/{metadata_path.name}")
print("Download and save this Kaggle output:", archive_path)